# Word2Vec + Logistic Regression Classifier

This notebook uses the architecture from the [official TensorFlow Word2Vec Tutorial](https://www.tensorflow.org/text/tutorials/word2vec) to train skip-gram word embeddings from scratch. 

After extracting the learned embeddings, we average the word vectors for each sequence to create sentence representations and classify them using a Logistic Regression model.

In [20]:
import io
import re
import string
import tqdm
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Setup standard seeds and configurations
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE
tf.random.set_seed(SEED)

## 1. Load the Dataset
Loading `train_noDuplicates.csv`. 
*(Note: Be sure to change `'text'` and `'label'` below if your dataset uses different column names).*

In [26]:
# Load the explicitly named dataset
df = pd.read_csv('/kaggle/input/datasets/juliaduboisas/train-noduplicates/train_noDuplicates.csv')

# Drop any rows with missing data in our target columns
df = df.dropna(subset=['resp_text', 'clarity'])
df = df.drop(columns=['duplicatedText'])

text_data = df['resp_text'].astype(str).tolist()
labels = df['clarity'].tolist()

print(f"Loaded {len(text_data)} samples.")

Loaded 18285 samples.


## 2. Text Preprocessing & Vectorization
We standardize, tokenize, and map words to integer IDs using Keras `TextVectorization`.

In [27]:
# Define vectorization parameters
vocab_size = 10000
sequence_length = 50

# Custom standardizer to lowercase and remove punctuation
def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    return tf.strings.regex_replace(lowercase, '[%s]' % re.escape(string.punctuation), '')

vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length
)

# Build the vocabulary based on the dataset
vectorize_layer.adapt(text_data)
inverse_vocab = vectorize_layer.get_vocabulary()
print(f"Vocabulary size: {len(inverse_vocab)}")

# Vectorize all text data into sequences of integers
text_vectorized = vectorize_layer(text_data).numpy()

Vocabulary size: 10000


## 3. Generate Skip-grams & Negative Samples
Using the candidate sampling approach shown in the tutorial to prepare `(target, context)` pairs for training.

In [29]:
def generate_training_data(sequences, window_size, num_ns, vocab_size, seed):
    targets, contexts, w2v_labels = [], [], []
    
    # tqdm will give us a nice progress bar
    for sequence in tqdm.tqdm(sequences):
        # Generate positive skip-gram pairs
        positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
              sequence,
              vocabulary_size=vocab_size,
              window_size=window_size,
              negative_samples=0,
              seed=42)
        
        for target_word, context_word in positive_skip_grams:
            context_class = tf.expand_dims(tf.constant([context_word], dtype="int64"), 1)
            
            # Generate negative samples
            negative_sampling_candidates, _, _ = tf.random.log_uniform_candidate_sampler(
                true_classes=context_class,
                num_true=1,
                num_sampled=num_ns,
                unique=True,
                range_max=vocab_size,
                seed=seed,
                name="negative_sampling")
            
            # Build context and label vectors
            context = tf.concat([tf.squeeze(context_class, 1), negative_sampling_candidates], 0)
            label = tf.constant([1] + [0]*num_ns, dtype="int64")
            
            targets.append(target_word)
            contexts.append(context)
            w2v_labels.append(label)

    return targets, contexts, w2v_labels

# Generate Word2Vec training data
window_size = 2
num_ns = 4

targets, contexts, w2v_labels = generate_training_data(
    sequences=text_vectorized, 
    window_size=window_size, 
    num_ns=num_ns, 
    vocab_size=vocab_size, 
    seed=SEED
)

# Convert to tensors
targets = np.array(targets)
contexts = np.array(contexts)
w2v_labels = np.array(w2v_labels)

# Build performant tf.data.Dataset
BATCH_SIZE = 1024
BUFFER_SIZE = 10000
dataset = tf.data.Dataset.from_tensor_slices(((targets, contexts), w2v_labels))
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)
dataset = dataset.cache().prefetch(buffer_size=AUTOTUNE)

100%|██████████| 18285/18285 [49:14<00:00,  6.19it/s]  


## 4. Build and Train the Subclassed Word2Vec Model
This defines the `Word2Vec` model and computes the dot products between target and context embeddings.

In [30]:
class Word2Vec(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, num_ns):
        super(Word2Vec, self).__init__()
        self.target_embedding = layers.Embedding(vocab_size,
                                                 embedding_dim,
                                                 input_length=1,
                                                 name="w2v_embedding")
        self.context_embedding = layers.Embedding(vocab_size,
                                                  embedding_dim,
                                                  input_length=num_ns+1)

    def call(self, pair):
        target, context = pair
        if len(target.shape) == 2:
            target = tf.squeeze(target, axis=1)
            
        word_emb = self.target_embedding(target)
        context_emb = self.context_embedding(context)
        
        # Calculate dot product
        dots = tf.einsum('be,bce->bc', word_emb, context_emb)
        return dots

embedding_dim = 128
word2vec = Word2Vec(vocab_size, embedding_dim, num_ns)

word2vec.compile(optimizer='adam',
                 loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])

# Train the Word2Vec model 
# (You may want to increase epochs depending on your dataset size)
word2vec.fit(dataset, epochs=5)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


  36/3052 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.2547 - loss: 1.6067 

I0000 00:00:1790788654.642587     139 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3052/3052 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.5763 - loss: 1.0881
Epoch 2/5
3052/3052 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.6555 - loss: 0.8988
Epoch 3/5
3052/3052 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.6777 - loss: 0.8395
Epoch 4/5
3052/3052 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.6915 - loss: 0.8024
Epoch 5/5
3052/3052 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.7018 - loss: 0.7751


## 5. Feature Engineering: Word Vector Averaging
We extract the learned weights from the `'w2v_embedding'` layer. Then, we average the word vectors for the tokens in each sentence to create a single vector representation for classification.

In [31]:
# Retrieve the learned word embeddings
weights = word2vec.get_layer('w2v_embedding').get_weights()[0]

def get_sentence_vector(sequence):
    # Ignore 0 (padding tokens) so they don't skew the average
    valid_words = [w for w in sequence if w != 0]
    
    if len(valid_words) == 0:
        return np.zeros(embedding_dim)
    
    # Retrieve embedding for each word and calculate mean over axis 0
    embeddings = [weights[word] for word in valid_words]
    return np.mean(embeddings, axis=0)

# Create final feature matrix X
X = np.array([get_sentence_vector(seq) for seq in text_vectorized])
y = np.array(labels)

print(f"Feature matrix X shape: {X.shape}")
print(f"Target array y shape: {y.shape}")

Feature matrix X shape: (18285, 128)
Target array y shape: (18285,)


## 6. Classification with Logistic Regression
We perform a train/test split on our sentence vectors and feed them into Scikit-Learn's `LogisticRegression`.

In [32]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

# Predict and Evaluate
y_pred = lr_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.41318020235165437

Classification Report:
               precision    recall  f1-score   support

          c1       0.42      0.36      0.39      1098
        c234       0.37      0.35      0.36      1282
          c5       0.44      0.52      0.48      1277

    accuracy                           0.41      3657
   macro avg       0.41      0.41      0.41      3657
weighted avg       0.41      0.41      0.41      3657

